# Environment Setup

This section installs required libraries, imports dependencies, defines project configuration parameters, and initializes the SageMaker environment.

In [1]:
# Configure project environment

# !pip uninstall sagemaker
!pip install -q sagemaker==2.224.4 awswrangler

import warnings
warnings.filterwarnings("ignore")

import boto3
import sagemaker
import pandas as pd
import numpy as np

from sagemaker.session import Session
from sagemaker import get_execution_role
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TARGET_COLUMN = "TARGET"
SAMPLE_SIZE_PER_CLASS = 10000
FEATURE_GROUP_NAME = "loan-approval-features"
PROJECT_PREFIX = "loanwise"

RAW_DATA_PATH = r"data/application_train.csv"

sess = Session()
role = get_execution_role()
bucket = sess.default_bucket()
region = boto3.Session().region_name

print(f"SageMaker SDK Version : {sagemaker.__version__}")
print(f"Region                : {region}")
print(f"Bucket                : {bucket}")
print(f"Feature Group         : {FEATURE_GROUP_NAME}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


SageMaker SDK Version : 2.224.4
Region                : us-east-1
Bucket                : sagemaker-us-east-1-612256167011
Feature Group         : loan-approval-features


In [2]:
import sagemaker
print(sagemaker.__version__)
print(type(sagemaker))

<class 'module'>


# Load Dataset

Load the Home Credit Default Risk training dataset and inspect its overall structure.

In [3]:
# Load Home Credit training dataset

df = pd.read_csv(RAW_DATA_PATH)

print(f"Dataset Shape : {df.shape}")

display(df.head())

display(df[["TARGET"]].value_counts())

Dataset Shape : (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


TARGET
0         282686
1          24825
Name: count, dtype: int64

# Create Balanced Dataset

To satisfy project requirements while minimizing infrastructure costs, create a balanced dataset containing 10,000 records from each target class.

In [4]:
# Create balanced dataset

class_0 = df[df[TARGET_COLUMN] == 0].sample(
    n=SAMPLE_SIZE_PER_CLASS,
    random_state=RANDOM_STATE
)

class_1 = df[df[TARGET_COLUMN] == 1].sample(
    n=SAMPLE_SIZE_PER_CLASS,
    random_state=RANDOM_STATE
)

df = pd.concat([class_0, class_1])

df = df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

df.to_csv(
    "balanced_dataset.csv",
    index=False
)

print(f"Balanced Dataset Shape : {df.shape}")
print(df[TARGET_COLUMN].value_counts())

Balanced Dataset Shape : (20000, 122)
TARGET
1    10000
0    10000
Name: count, dtype: int64


# Exploratory Data Analysis

Review dataset structure, data types, summary statistics, and target distribution.

In [5]:
# Perform exploratory data analysis

print(df.info())

display(df.describe().T)

target_distribution = (
    df[TARGET_COLUMN]
    .value_counts(normalize=True)
    .mul(100)
    .rename("Percentage")
)

display(target_distribution)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), object(16)
memory usage: 18.6+ MB
None


,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,20000.0,277718.925850,102438.641787,100002.0,189594.5,277195.5,366491.5,456255.0
TARGET,20000.0,0.500000,0.500013,0.0,0.0,0.5,1.0,1.0
CNT_CHILDREN,20000.0,0.435500,0.734756,0.0,0.0,0.0,1.0,11.0
AMT_INCOME_TOTAL,20000.0,165049.494975,92579.557409,25650.0,112500.0,140625.0,202500.0,3150000.0
AMT_CREDIT,20000.0,581969.178900,380740.559135,45000.0,276277.5,508495.5,781920.0,3020760.0
...,...,...,...,...,...,...,...,...
AMT_REQ_CREDIT_BUREAU_DAY,16909.0,0.007156,0.101484,0.0,0.0,0.0,0.0,4.0
AMT_REQ_CREDIT_BUREAU_WEEK,16909.0,0.035366,0.200963,0.0,0.0,0.0,0.0,6.0
AMT_REQ_CREDIT_BUREAU_MON,16909.0,0.236679,0.780530,0.0,0.0,0.0,0.0,16.0
AMT_REQ_CREDIT_BUREAU_QRT,16909.0,0.264652,0.641254,0.0,0.0,0.0,0.0,19.0


TARGET
1    50.0
0    50.0
Name: Percentage, dtype: float64

# Data Quality Assessment

Assess missing values and duplicate records before feature engineering.

In [6]:
# Assess data quality

missing_df = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percent": (
        df.isnull().sum() / len(df)
    ).mul(100).round(2)
})

missing_df = missing_df.sort_values(
    "Missing_Percent",
    ascending=False
)

duplicate_count = df.duplicated().sum()

display(missing_df.head(20))

print(f"Duplicate Records : {duplicate_count}")

,Missing_Count,Missing_Percent
COMMONAREA_MEDI,14426,72.13
COMMONAREA_AVG,14426,72.13
COMMONAREA_MODE,14426,72.13
NONLIVINGAPARTMENTS_AVG,14335,71.68
NONLIVINGAPARTMENTS_MODE,14335,71.68
NONLIVINGAPARTMENTS_MEDI,14335,71.68
FONDKAPREMONT_MODE,14188,70.94
LIVINGAPARTMENTS_MEDI,14158,70.79
LIVINGAPARTMENTS_MODE,14158,70.79
LIVINGAPARTMENTS_AVG,14158,70.79


Duplicate Records : 0


# Feature Engineering

Select relevant business features, handle missing values, encode categorical variables, and create an event timestamp required by SageMaker Feature Store.

In [7]:
# Engineer model features

selected_features = [
    "SK_ID_CURR",
    "TARGET",
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "CNT_CHILDREN",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "CNT_FAM_MEMBERS",
    "REGION_POPULATION_RELATIVE",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]

feature_df = df[selected_features].copy()

numeric_cols = feature_df.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_cols = feature_df.select_dtypes(
    include=["object"]
).columns

feature_df[numeric_cols] = (
    feature_df[numeric_cols]
    .fillna(feature_df[numeric_cols].median())
)

feature_df[categorical_cols] = (
    feature_df[categorical_cols]
    .fillna("Unknown")
)

feature_df = pd.get_dummies(
    feature_df,
    columns=[
        "CODE_GENDER",
        "FLAG_OWN_CAR",
        "FLAG_OWN_REALTY"
    ],
    drop_first=True,
    dtype="int64"
)

feature_df["event_time"] = (
    pd.Timestamp.utcnow()
    .strftime("%Y-%m-%dT%H:%M:%SZ")
)

print(f"Feature Dataset Shape : {feature_df.shape}")

display(feature_df.head())

Feature Dataset Shape : (20000, 18)


,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,DAYS_BIRTH,DAYS_EMPLOYED,CNT_FAM_MEMBERS,REGION_POPULATION_RELATIVE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,CODE_GENDER_M,FLAG_OWN_CAR_Y,FLAG_OWN_REALTY_Y,event_time
0,237856,1,0,112500.0,355536.0,18283.5,270000.0,-15355,-1343,2.0,0.046220,0.440128,0.667438,0.357293,0,0,1,2026-06-13T11:39:45Z
1,352956,0,0,63000.0,675000.0,19867.5,675000.0,-22202,-4192,2.0,0.031329,0.888944,0.785819,0.885488,0,0,1,2026-06-13T11:39:45Z
2,426282,0,0,607500.0,585000.0,29250.0,585000.0,-12799,-3316,1.0,0.046220,0.440128,0.718854,0.076984,0,1,1,2026-06-13T11:39:45Z
3,427360,0,0,157500.0,463626.0,23800.5,387000.0,-14767,-514,2.0,0.030755,0.440128,0.650402,0.468660,1,0,1,2026-06-13T11:39:45Z
4,176109,1,0,270000.0,898326.0,29110.5,643500.0,-19620,-3832,2.0,0.016612,0.566862,0.652443,0.245851,0,0,1,2026-06-13T11:39:45Z


In [8]:
# Validate Feature Store compatibility

print(feature_df.dtypes)

unsupported_types = feature_df.dtypes[
    ~feature_df.dtypes.astype(str).isin(
        ["int64", "float64", "object"]
    )
]

display(unsupported_types)

SK_ID_CURR                      int64
TARGET                          int64
CNT_CHILDREN                    int64
AMT_INCOME_TOTAL              float64
AMT_CREDIT                    float64
AMT_ANNUITY                   float64
AMT_GOODS_PRICE               float64
DAYS_BIRTH                      int64
DAYS_EMPLOYED                   int64
CNT_FAM_MEMBERS               float64
REGION_POPULATION_RELATIVE    float64
EXT_SOURCE_1                  float64
EXT_SOURCE_2                  float64
EXT_SOURCE_3                  float64
CODE_GENDER_M                   int64
FLAG_OWN_CAR_Y                  int64
FLAG_OWN_REALTY_Y               int64
event_time                     object
dtype: object


Series([], dtype: object)

# Create Training Artifacts

Split the feature-engineered dataset into training and testing datasets and save them locally.

In [9]:
# Create training artifacts

train_df, test_df = train_test_split(
    feature_df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=feature_df[TARGET_COLUMN]
)

train_df.to_csv(
    "processed_train.csv",
    index=False
)

test_df.to_csv(
    "processed_test.csv",
    index=False
)

print(f"Training Dataset Shape : {train_df.shape}")
print(f"Testing Dataset Shape  : {test_df.shape}")

Training Dataset Shape : (16000, 18)
Testing Dataset Shape  : (4000, 18)


# Upload Artifacts to Amazon S3

Upload balanced, training, and testing datasets to Amazon S3 for downstream processing and model development.

In [10]:
# Upload project datasets to Amazon S3

balanced_s3_path = sess.upload_data(
    path="balanced_dataset.csv",
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/data"
)

train_s3_path = sess.upload_data(
    path="processed_train.csv",
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/train"
)

test_s3_path = sess.upload_data(
    path="processed_test.csv",
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/test"
)

print("Balanced Dataset")
print(balanced_s3_path)

print("\nTraining Dataset")
print(train_s3_path)

print("\nTesting Dataset")
print(test_s3_path)

Balanced Dataset
s3://sagemaker-us-east-1-612256167011/loanwise/data/balanced_dataset.csv

Training Dataset
s3://sagemaker-us-east-1-612256167011/loanwise/train/processed_train.csv

Testing Dataset
s3://sagemaker-us-east-1-612256167011/loanwise/test/processed_test.csv


# Create SageMaker Feature Store

Create a Feature Group, define feature metadata, provision the online and offline stores, and ingest training records.

In [11]:
# Create SageMaker Feature Store

from sagemaker.feature_store.feature_group import FeatureGroup
from botocore.exceptions import ClientError
from time import sleep

sm_client = boto3.client("sagemaker")

feature_group = FeatureGroup(
    name=FEATURE_GROUP_NAME,
    sagemaker_session=sess
)

feature_group_exists = False

try:
    sm_client.describe_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME
    )

    feature_group_exists = True

    print(
        f"Feature Group '{FEATURE_GROUP_NAME}' already exists."
    )

except ClientError:

    print(
        f"Creating Feature Group '{FEATURE_GROUP_NAME}'..."
    )

if not feature_group_exists:

    feature_group.load_feature_definitions(
        data_frame=train_df
    )

    feature_group.create(
        s3_uri=f"s3://{bucket}/feature-store/",
        record_identifier_name="SK_ID_CURR",
        event_time_feature_name="event_time",
        role_arn=role,
        enable_online_store=True
    )

    max_attempts = 20

    for attempt in range(max_attempts):

        status = feature_group.describe()[
            "FeatureGroupStatus"
        ]

        print(
            f"Attempt {attempt + 1}/{max_attempts} "
            f"| Status: {status}"
        )

        if status == "Created":
            print("Feature Group created successfully.")
            break

        if status in [
            "CreateFailed",
            "DeleteFailed"
        ]:

            raise Exception(
                f"Feature Group creation failed. "
                f"Status: {status}"
            )

        sleep(30)

    else:

        raise TimeoutError(
            "Feature Group creation exceeded "
            "10 minutes."
        )

    feature_group.ingest(
        data_frame=train_df,
        max_workers=3,
        wait=True
    )

    print("Feature ingestion completed.")

else:

    status = sm_client.describe_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME
    )["FeatureGroupStatus"]

    print(f"Current Status: {status}")

    print(
        "Skipping Feature Store ingestion because "
        "the Feature Group already exists."
    )

display(feature_group.describe())

Creating Feature Group 'loan-approval-features'...


Attempt 1/20 | Status: Creating


Attempt 2/20 | Status: Created
Feature Group created successfully.


Feature ingestion completed.


{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:612256167011:feature-group/loan-approval-features',
 'FeatureGroupName': 'loan-approval-features',
 'RecordIdentifierFeatureName': 'SK_ID_CURR',
 'EventTimeFeatureName': 'event_time',
 'FeatureDefinitions': [{'FeatureName': 'SK_ID_CURR',
   'FeatureType': 'Integral'},
  {'FeatureName': 'TARGET', 'FeatureType': 'Integral'},
  {'FeatureName': 'CNT_CHILDREN', 'FeatureType': 'Integral'},
  {'FeatureName': 'AMT_INCOME_TOTAL', 'FeatureType': 'Fractional'},
  {'FeatureName': 'AMT_CREDIT', 'FeatureType': 'Fractional'},
  {'FeatureName': 'AMT_ANNUITY', 'FeatureType': 'Fractional'},
  {'FeatureName': 'AMT_GOODS_PRICE', 'FeatureType': 'Fractional'},
  {'FeatureName': 'DAYS_BIRTH', 'FeatureType': 'Integral'},
  {'FeatureName': 'DAYS_EMPLOYED', 'FeatureType': 'Integral'},
  {'FeatureName': 'CNT_FAM_MEMBERS', 'FeatureType': 'Fractional'},
  {'FeatureName': 'REGION_POPULATION_RELATIVE', 'FeatureType': 'Fractional'},
  {'FeatureName': 'EXT_SOURCE_1', 'F

# Verify Feature Store and Project Artifacts

Review Feature Store configuration and verify all generated project artifacts.

In [12]:
# Verify project artifacts

display(feature_group.describe())

display(feature_group.feature_definitions)

print("\nGenerated Artifacts")
print("-------------------")
print("balanced_dataset.csv")
print("processed_train.csv")
print("processed_test.csv")
print(FEATURE_GROUP_NAME)

print("\nS3 Artifacts")
print("------------")
print(balanced_s3_path)
print(train_s3_path)
print(test_s3_path)

{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:612256167011:feature-group/loan-approval-features',
 'FeatureGroupName': 'loan-approval-features',
 'RecordIdentifierFeatureName': 'SK_ID_CURR',
 'EventTimeFeatureName': 'event_time',
 'FeatureDefinitions': [{'FeatureName': 'SK_ID_CURR',
   'FeatureType': 'Integral'},
  {'FeatureName': 'TARGET', 'FeatureType': 'Integral'},
  {'FeatureName': 'CNT_CHILDREN', 'FeatureType': 'Integral'},
  {'FeatureName': 'AMT_INCOME_TOTAL', 'FeatureType': 'Fractional'},
  {'FeatureName': 'AMT_CREDIT', 'FeatureType': 'Fractional'},
  {'FeatureName': 'AMT_ANNUITY', 'FeatureType': 'Fractional'},
  {'FeatureName': 'AMT_GOODS_PRICE', 'FeatureType': 'Fractional'},
  {'FeatureName': 'DAYS_BIRTH', 'FeatureType': 'Integral'},
  {'FeatureName': 'DAYS_EMPLOYED', 'FeatureType': 'Integral'},
  {'FeatureName': 'CNT_FAM_MEMBERS', 'FeatureType': 'Fractional'},
  {'FeatureName': 'REGION_POPULATION_RELATIVE', 'FeatureType': 'Fractional'},
  {'FeatureName': 'EXT_SOURCE_1', 'F

[FeatureDefinition(feature_name='SK_ID_CURR', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='TARGET', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='CNT_CHILDREN', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='AMT_INCOME_TOTAL', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='AMT_CREDIT', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='AMT_ANNUITY', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='AMT_GOODS_PRICE', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='DAYS_BIRTH', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None)


Generated Artifacts
-------------------
balanced_dataset.csv
processed_train.csv
processed_test.csv
loan-approval-features

S3 Artifacts
------------
s3://sagemaker-us-east-1-612256167011/loanwise/data/balanced_dataset.csv
s3://sagemaker-us-east-1-612256167011/loanwise/train/processed_train.csv
s3://sagemaker-us-east-1-612256167011/loanwise/test/processed_test.csv
